## Собираем информацию для дальнейшнего анализа предложений на адекватность перевода

Выполняется по [задаче](https://tsa04.isa.ru/ibuyanov/rudeft/-/issues/12)


### Алгоритм

**Задача** - семплировать 600 текстов без IOB разметки, но с русским переводом. Семплировать необходимо так, чтобы тематики были покрыты равномерно. Тематики можно определить по названию файла в соответствующем поле hf датасета.

1. Из HF датасета собираем те, где есть русский текст
2. Вычитаем те, где есть связи (т.к. они уже на разметке и валидации). 
Получаем предложения, где есть русский текст и нет разметки и нет IOB разметки.
3. Вычитаем строки, где длина меньше 50 символов (???), чтобы отмести короткие предложения по типу "21. " "345. " (т.е. где были номера просто, без текста)
    - методом визуального просмотра было определено, что в предложениях до 50 символов встречается мало полезной информации.
4. Анализируем тематики и собираем 600 примеров (точнее по 200 на каждого) равномерно
    - тематики в названии файла. Тематика начинается с букв `/t1_...`, `/t2_...` и т.д.
5. получаем HF датасет из 600 строк

6. Конвертируем его к формату Label Studio
    - перед этим настраиваем интерфейс LS, чтобы знать к какому формату подогнать
    - конвертируем
    - загружаем в LS

In [1]:
import json
from tqdm import tqdm
import logging

In [88]:
import random
from collections import defaultdict

Настройка логгера, для вывода подробной информации работа скрипта в файл `log.log`

При этом основной вывод прогрессбара от `tqdm` будет осуществляться в `Out` ячейку

In [2]:
# Настройка логгера для записи в файл
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Создание обработчика для записи в файл
file_handler = logging.FileHandler('log.log')
file_handler.setLevel(logging.DEBUG)

# Создание форматтера для сообщений
FORMAT = "%(asctime)s %(levelname)s [%(process)s]: %(name)s: %(message)s"
formatter = logging.Formatter(FORMAT)
file_handler.setFormatter(formatter)

# Добавление обработчика к логгеру
logger.addHandler(file_handler)

Предполагается, что оригинальный датасет лежит в директории `../hf_deft_ru/`

In [3]:
# orig files
dev_file = "../hf_deft_ru/dev.json"
train_file = "../hf_deft_ru/train.json"

In [46]:
def get_hf_with_only_rus(hf_file):
    """ Функция получение датасета HF (Hugging Face) , но только с русским текстом и отсутсвующей разметкой

    :param hf_file: путь к файлу датасета в HF формате
    :return filtered_lines_hf: список отфильтрованных строк из HF файла
    
    """
    cnt_processed = 0
    cnt_added_in_ls = 0
    cnt_skip_span_relation = 0
    cnt_skip_text_rus = 0
    cnt_skip_all = 0


    with open(hf_file, 'r') as f:
        lines = f.readlines()
    
    filtered_lines_hf = []
    
    for line in tqdm(lines):
        data = json.loads(line)
        cnt_processed += 1
        # 1. Проверка на наличие русского языка
        # 3. Проверка на длину русского текста (должна быть больше 50 символов)
        N = 50
        if "text_rus" in data and data["text_rus"] != "" and len(data["text_rus"]) > N:
            # 2. Проверка на отсутствие спанов или связей
            if ( "spans" in data and len(data["spans"]) == 0 ) and ( "relations" in data and len(data["relations"]) == 0 ):
                filtered_lines_hf.append(line)
                logging.info(f"ID: '{data.get('id', 'no_id')}' successfully added in new filtered HF dataset!")
                cnt_added_in_ls += 1
            else:
                logging.info(f"ID: '{data.get('id', 'no_id')}' has spans or relations! Skip!")
                cnt_skip_span_relation +=1
                cnt_skip_all += 1
        else:
            logging.info(f"ID: '{data.get('id', 'no_id')}' has not text_rus or text_rus is empty or len(text_rus) < {N}! Skip!")
            cnt_skip_text_rus += 1
            cnt_skip_all += 1

    logging.info(f"Processed: {cnt_processed}, \tAdded: {cnt_added_in_ls}, \tSkip_text_rus: {cnt_skip_text_rus}, \tSkip_span_rel: {cnt_skip_span_relation}, \tSkip_total: {cnt_skip_all}.")

    return filtered_lines_hf


In [72]:
filtered_lines_hf_dev = get_hf_with_only_rus(dev_file)
filtered_lines_hf_train = get_hf_with_only_rus(train_file)

100%|██████████| 1179/1179 [00:00<00:00, 6803.23it/s]


100%|██████████| 24184/24184 [00:03<00:00, 6763.31it/s]


In [73]:
!grep "Processed:" log.log 

2024-09-25 15:09:10,848 INFO [183611]: root: Processed: 1179, 	Added: 525, 	Skip_text_rus: 322, 	Skip_span_rel: 332, 	Skip_total: 654.
2024-09-25 15:09:13,892 INFO [183611]: root: Processed: 24184, 	Added: 1481, 	Skip_text_rus: 21980, 	Skip_span_rel: 723, 	Skip_total: 22703.
2024-09-25 15:21:21,444 INFO [183611]: root: Processed: 1179, 	Added: 485, 	Skip_text_rus: 369, 	Skip_span_rel: 325, 	Skip_total: 694.
2024-09-25 15:21:24,376 INFO [183611]: root: Processed: 24184, 	Added: 1389, 	Skip_text_rus: 22081, 	Skip_span_rel: 714, 	Skip_total: 22795.
2024-09-25 15:36:11,496 INFO [183611]: root: Processed: 1179, 	Added: 485, 	Skip_text_rus: 369, 	Skip_span_rel: 325, 	Skip_total: 694.
2024-09-25 15:36:15,122 INFO [183611]: root: Processed: 24184, 	Added: 1389, 	Skip_text_rus: 22081, 	Skip_span_rel: 714, 	Skip_total: 22795.


In [74]:
filtered_lines_hf_all = filtered_lines_hf_dev.copy()
filtered_lines_hf_all.extend(filtered_lines_hf_train)

In [75]:
print(f"filtered   dev : {len(filtered_lines_hf_dev)}")
print(f"filtered train : {len(filtered_lines_hf_train)}")
print(f"filtered   all : {len(filtered_lines_hf_all)}")

filtered   dev : 485
filtered train : 1389
filtered   all : 1874


А уменьшенный датасет сохраняем в директорию `./deft_with_only_rus/`

In [78]:
out_file_hf = "./deft_with_only_rus/data.json"
with open(out_file_hf, 'w', encoding='utf-8') as f:
    f.write(''.join(filtered_lines_hf_all))

In [69]:
def make_label_studio_obj_for_translation(data: dict):
    """ Функция обработки одного элемента датасета для перевода.

    Создается элемент для добавления в Label Studio для проверки корректности перевода. 
    Преобразовываются к нужному виду все поля:
    - предложения на русском и на английском языках - к формату `data`,
    - внутренний id датасета - к формату `meta`,
    
    :param data: загруженный словарь данных об элементе датасета.
    :return label_studio_obj: преобразованный к формату Label Studio объект данных.
    
    """
    
    # Создаем объект для Label Studio
    label_studio_obj = {
        "meta": {
            "id": data["id"]
        },
        "data": {
            "text": data["text"],
            "text_rus": data.get("text_rus", "")
        },
        "predictions": [],
    }

        
    return label_studio_obj

In [90]:
def create_balanced_sets(dataset, num_sets, set_size):
    """ Создание сбалансированных наборов данных

    :param dataset: датасет (список из словарей)
    :param num_sets: количество наборов, на которые необходимо разделить
    :param set_size: количество элементов в одном наборе
    :return sets: список из сбалансированных наборов

    
    """
    # Подсчет количества примеров каждой тематики
    topic_counts = defaultdict(int)
    for example in dataset:
        topic_counts[example['subj']] += 1

    # Расчет соотношения тематик
    total_examples = len(dataset)
    topic_ratios = {topic: count / total_examples for topic, count in topic_counts.items()}
    print(f"topic_ratios: (распределение по предметам в процентном соотношении)")
    print(topic_ratios)
    # Создание пустых наборов
    sets = [[] for _ in range(num_sets)]

    # Разделение примеров по тематикам
    topic_examples = defaultdict(list)
    for example in dataset:
        topic_examples[example['subj']].append(example)


    # Заполнение наборов
    for topic, examples in topic_examples.items():
        random.shuffle(examples)
        ratio = topic_ratios[topic]
        num_examples_per_set = int(ratio * set_size)
        
        for i in range(num_sets):
            sets[i].extend(examples[i * num_examples_per_set:(i + 1) * num_examples_per_set])

    # Если какой-то набор меньше set_size, добавим случайные примеры
    for i in range(num_sets):
        while len(sets[i]) < set_size:
            topic = random.choice(list(topic_examples.keys()))
            if topic_examples[topic]:
                sets[i].append(topic_examples[topic].pop())
            else:
                # Удаляем тему, если не осталось примеров
                del topic_examples[topic]

    return sets


In [91]:
def stratificate_filtered_lines_hf_by_subject(filtered_lines_hf, num_sets = 3, set_size = 200):
    """ Функция стратификации данных по тематикам.

    :param filtered_lines_hf: список строк отфильтрованного датасета в HF формате
    :param num_sets: количество наборов, на которые необходимо разделить
    :param set_size: количество элементов в одном наборе
    :return balanced_sets: список из сбалансированных наборов
    
    """
    # stratificated_dataset = []
    filtered_dataset = []
    for line in filtered_lines_hf:
        data = json.loads(line)
        source = data["source_file"]
        name = source.split("/")[2]
        subj = name[:2]
        # print(subj)
        data['subj'] = subj
        filtered_dataset.append(data)

    balanced_sets = create_balanced_sets(filtered_dataset, num_sets, set_size)
    # for balanced_set in balanced_sets:
    #     for elem in balanced_set:
    #         elem.pop('subj')


    return balanced_sets
    # return stratificated_dataset

In [95]:
balanced_sets = stratificate_filtered_lines_hf_by_subject(filtered_lines_hf_all, set_size=200)

topic_ratios
{'t1': 0.10992529348986126, 't2': 0.0832443970117396, 't3': 0.047491995731056565, 't4': 0.10405549626467449, 't5': 0.3633938100320171, 't6': 0.021344717182497332, 't7': 0.2705442902881537}


In [96]:
for i, balanced_set in enumerate(balanced_sets):
    print(f"Set {i+1}: {len(balanced_set)} examples")
    for topic in sorted(set(example['subj'] for example in balanced_set)):
        count = sum(1 for example in balanced_set if example['subj'] == topic)
        print(f"  {topic}: {count} examples")

Set 1: 200 examples
  t1: 21 examples
  t2: 16 examples
  t3: 9 examples
  t4: 21 examples
  t5: 72 examples
  t6: 4 examples
  t7: 57 examples
Set 2: 200 examples
  t1: 21 examples
  t2: 17 examples
  t3: 10 examples
  t4: 21 examples
  t5: 72 examples
  t6: 4 examples
  t7: 55 examples
Set 3: 200 examples
  t1: 22 examples
  t2: 16 examples
  t3: 9 examples
  t4: 21 examples
  t5: 73 examples
  t6: 5 examples
  t7: 54 examples


In [97]:
out_dir = "./hf_deft_sets/"
for i, balanced_set in enumerate(balanced_sets):
    for elem in balanced_set:
        elem.pop('subj')
    out_file_path = f"{out_dir}dataset{i}.json"

    new_dataset = []
    for hf_elem in balanced_set:
        new_dataset.append(json.dumps(hf_elem))
    with open(out_file_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(new_dataset))

In [100]:
def convert_filtered_hf_to_ls(hf_file, ls_file):
    """ Функция конвертации датасета HF (Hugging Face) в LS (Label Studio) формат.

    :param hf_file: путь к файлу датасета в HF формате
    :param ls_file: путь к файлу для сохранения преобразованного датасета в LS формате
    
    """

    with open(hf_file, 'r') as f:
        lines = f.readlines()

    label_studio_data = []

    for line in tqdm(lines):
        data = json.loads(line)
        label_studio_obj = make_label_studio_obj_for_translation(data)
        label_studio_data.append(label_studio_obj)

    with open(ls_file, 'w') as f:
        json.dump(label_studio_data, f, indent=4)


In [103]:
for i in range(3):
    hf_file = f"./hf_deft_sets/dataset{i}.json"
    ls_file = f"./ls_deft_sets/dataset{i}.json"
    convert_filtered_hf_to_ls(hf_file, ls_file)


100%|██████████| 200/200 [00:00<00:00, 49766.30it/s]
